# Notebook 04 — Modeling: Disaster-Level Classification (Strategic / Budget Forecasting)
**Label:** `funding_tier` — 4 classes based on total PA obligations per disaster:
- 0 = Minor (<$1M) | 1 = Moderate ($1M–$50M) | 2 = Major ($50M–$500M) | 3 = Catastrophic (>$500M)

**Use case:** FEMA budget planning — classify total spend tier before projects are filed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import sys
sys.path.append('../')
from utils import classification_metrics, time_based_split, DISASTER_LABELS

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

PROCESSED = '../data/processed/'
disas = pd.read_csv(PROCESSED + 'cleaned_disaster_level.csv', low_memory=False)
print('Disaster-level shape:', disas.shape)
print('Target distribution:\n', disas['funding_tier'].value_counts().sort_index())

## 4.1 Define Features & Target


In [ ]:
CAT_FEATURES = ['incidentType', 'stateAbbreviation', 'incident_season']
NUM_FEATURES = [
    'declaration_lag_days',
    'incident_duration_days',
    'n_counties',
    'n_projects',
    'prior_disasters_5yr',
    'population',
    'median_income',
    'poverty_rate',
    'risk_score',
]
TARGET = 'funding_tier'

CAT_FEATURES = [c for c in CAT_FEATURES if c in disas.columns]
NUM_FEATURES = [c for c in NUM_FEATURES if c in disas.columns]
FEATURES     = CAT_FEATURES + NUM_FEATURES

df_model = disas[FEATURES + [TARGET, 'incident_year']].dropna(subset=[TARGET])
df_model[TARGET] = df_model[TARGET].astype(int)
print(f'Modeling rows: {len(df_model):,}  |  Features: {len(FEATURES)}')
print('Categorical:', CAT_FEATURES)
print('Numeric:    ', NUM_FEATURES)
print('\nClass distribution:')
for t, n in df_model[TARGET].value_counts().sort_index().items():
    print(f'  Tier {t} {DISASTER_LABELS[t]:<28} {n:>5,}  ({100*n/len(df_model):.1f}%)')

## 4.2 Time-Based Train / Test Split
Train on disasters before 2018, test on 2018 onwards.
**Never use random split** on time-series data — it leaks future information into training.


In [11]:
SPLIT_YEAR = 2018
train = df_model[df_model['incident_year'] <  SPLIT_YEAR]
test  = df_model[df_model['incident_year'] >= SPLIT_YEAR]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train years: {train["incident_year"].min()} – {train["incident_year"].max()}')
print(f'Test  years: {test["incident_year"].min()}  – {test["incident_year"].max()}')


Train: 1,202  |  Test: 564
Train years: 1998 – 2017
Test  years: 2018  – 2026


## 4.3 Preprocessing Pipeline


In [12]:
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler())
])
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, CAT_FEATURES),
    ('num', num_pipe, NUM_FEATURES)
])
print('Preprocessor defined.')


Preprocessor defined.


## 4.4 Train & Evaluate All Models


In [ ]:
TARGET_NAMES = [DISASTER_LABELS[i] for i in range(4)]

models = {
    'Baseline (Stratified)': DummyClassifier(strategy='stratified', random_state=42),
    'Logistic Regression':   LogisticRegression(max_iter=1000, class_weight='balanced',
                                                random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                    random_state=42, n_jobs=-1),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=200, random_state=42),
}

results_disaster = {}
for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    m = classification_metrics(y_test.values, preds, label=name,
                               target_names=TARGET_NAMES)
    m['pipeline'] = pipe
    m['preds']    = preds
    results_disaster[name] = m

## 4.5 Results Summary Table


In [ ]:
summary = pd.DataFrame([
    {k: v for k, v in v.items() if k not in ('pipeline', 'preds')}
    for v in results_disaster.values()
]).set_index('label')
summary[['Accuracy', 'F1_weighted']]

## 4.6 Feature Importances — Random Forest


In [ ]:
rf_pipe  = results_disaster['Random Forest']['pipeline']
rf_model = rf_pipe.named_steps['model']
rf_pre   = rf_pipe.named_steps['pre']

ohe_names = rf_pre.transformers_[0][1].named_steps['ohe'].get_feature_names_out(CAT_FEATURES)
all_names = list(ohe_names) + NUM_FEATURES

importances = pd.Series(rf_model.feature_importances_, index=all_names)
importances.nlargest(20).sort_values().plot(
    kind='barh', figsize=(10, 6),
    title='Top 20 Feature Importances — Disaster-Level Random Forest Classifier',
    color='steelblue'
)
plt.tight_layout()
plt.savefig('../data/processed/feature_importance_disaster.png', dpi=150)
plt.show()

# Confusion matrix for best model
best_name = max(results_disaster, key=lambda k: results_disaster[k]['F1_weighted'])
cm = confusion_matrix(y_test, results_disaster[best_name]['preds'])
fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=TARGET_NAMES)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_name} (Disaster-Level)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix_disaster.png', dpi=150)
plt.show()

## 4.7 Save Best Pipeline


In [ ]:
best_name = max(results_disaster, key=lambda k: results_disaster[k]['F1_weighted'])
print(f'Best model: {best_name}  (F1_weighted = {results_disaster[best_name]["F1_weighted"]:.4f})')

with open(PROCESSED + 'best_disaster_model.pkl', 'wb') as f:
    pickle.dump({
        'pipeline':     results_disaster[best_name]['pipeline'],
        'X_test':       X_test,
        'y_test':       y_test,
        'preds':        results_disaster[best_name]['preds'],
        'features':     FEATURES,
        'cat_features': CAT_FEATURES,
        'num_features': NUM_FEATURES,
        'target_names': TARGET_NAMES,
        'model_name':   best_name,
        'level':        'disaster',
    }, f)
print('Saved best_disaster_model.pkl')